# 03 — Evaluation & Visualization

Load the best checkpoint, run on validation set, inspect per-keypoint PCK,
and overlay predictions on sample images.

In [ ]:
import sys
sys.path.insert(0, "../src")

import torch
import numpy as np
import yaml
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm

In [ ]:
from walker_gait.pipeline import WalkerGaitPipeline
from walker_gait.utils import draw_keypoints_on_image

model_cfg = yaml.safe_load(open("../configs/model.yaml"))
data_cfg = yaml.safe_load(open("../configs/data.yaml"))

## 1. Load fine-tuned pipeline

In [ ]:
pipeline = WalkerGaitPipeline(
    pose_model_name=model_cfg["backbone"]["name"],
    pose_checkpoint="../checkpoints/best",  # fine-tuned weights
    device="cuda" if torch.cuda.is_available() else "cpu",
)

## 2. Run on validation images and visualize

In [ ]:
import json

with open(data_cfg["paths"]["val_ann"]) as f:
    val_coco = json.load(f)

# Pick a few samples to visualize
sample_images = val_coco["images"][:8]
frames_dir = Path(data_cfg["paths"]["frames_dir"])

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, img_info in zip(axes.flat, sample_images):
    img_path = frames_dir / img_info["file_name"]
    image = Image.open(img_path).convert("RGB")
    result = pipeline(image)

    if len(result.keypoints) > 0:
        vis = draw_keypoints_on_image(
            np.array(image), result.keypoints, result.scores,
            score_threshold=0.3,
        )
        ax.imshow(vis)
    else:
        ax.imshow(image)
        ax.set_title("No detection")
    ax.axis("off")

plt.suptitle("Fine-Tuned ViTPose++ — Validation Samples", fontsize=14)
plt.tight_layout()
plt.show()

## 3. Per-keypoint PCK breakdown

Use this to identify which keypoints need more annotation focus.

In [ ]:
# TODO: compute per-keypoint PCK across the full validation set
# using walker_gait.training.metrics.compute_pck_per_keypoint
# and aggregate into a bar chart per keypoint name

# Placeholder — fill in once annotations exist
keypoint_names = data_cfg["keypoint_subset"]["names"]
# pck_per_kp = {...}
# plt.bar(keypoint_names, [pck_per_kp[n] for n in keypoint_names])
# plt.xticks(rotation=45, ha="right")
# plt.ylabel("PCK@0.05")
# plt.title("Per-Keypoint PCK on Validation Set")
# plt.tight_layout()
# plt.show()